# 1. Introduction

Ce notebook mesure la qualité des prévisions faites **en conditions réelles**, c'est-à-dire à partir des listes d'engagés publiées avant chaque course sur le site des start lists, en les comparant aux classements officiels de la FFC.

**Prévisions évaluées.** Les 35 paires (engagés, prévisions) rétroactives de 2026 stockées sur le bucket S3, une par semaine, horodatées les vendredis du 13 février au 9 octobre 2026 à 00:00 UTC (notebook `retroactive_forecasts_2026.ipynb`). Chaque course y est prévue par la fonction du paquet exécutée par le workflow programmé (`forecast_upcoming_entries`), avec l'information disponible la veille de la course.

**Différence avec l'évaluation de référence.** L'évaluation du notebook de modélisation (section 8.5) prend pour engagés d'une course passée ses coureurs classés : le vainqueur figure alors dans la liste de 10 dans 60,5 % des courses de 2026, contre 50,1 % pour la baseline. En conditions réelles, la liste des engagés diffère du classement officiel :

- des **engagés ne sont pas classés** : non-partants, abandons, coureurs arrivés au-delà des places classées ;
- des **classés ne sont pas engagés** : coureurs inscrits sur place, ou engagés que l'extraction n'a pas identifiés.

**Règles de la mesure.**

- Une course des engagés sans classement officiel rapproché (annulation, course regroupée avec l'Open, etc.) est écartée.
- Une course des engagés rapprochée de deux classements officiels (un tableau d'engagés unique pour deux courses, par exemple A1-A2 et A3-A4) est écartée : le modèle l'a prévue comme une seule course, avec un seul vainqueur.
- Les **classés non engagés sont retirés** du classement officiel, et le rang réel des autres est recalculé (les ex aequo gardent le même rang, le suivant saute) : le modèle n'est pas pénalisé pour un coureur qu'il ne pouvait pas connaître.
- Les **engagés non classés sont gardés**, sans aucune cible atteinte : ils reflètent l'incertitude propre à la course.
- Les rangs et les probabilités évalués sont ceux qui ont été publiés, calculés sur tous les engagés.
- Seules les courses de **plus de 20 classés engagés** sont évaluées : le classement dépasse ainsi largement le top 10, et un engagé non classé est bien hors des cibles.

**Métriques**, les mêmes que pour la référence (module `target_metrics`), calculées course par course puis moyennées :

- **rappel de la liste** d'une cible (victoire, podium, top 5, top 10) : part des coureurs ayant atteint la cible qui figurent dans la liste prévue (les 10 premiers prévus, 20 pour le top 10) ;
- **rang réciproque du vainqueur** : 1 / rang prévu du vainqueur s'il figure dans la liste de 10, 0 sinon ;
- **log-loss** et **score de Brier** des probabilités, sur toutes les lignes ;
- **calibration** : probabilité moyenne et fréquence observée par décile de probabilité.

**Plan.** Données (3), rapprochement avec les classements officiels (4), populations d'engagés et de classés (5), table évaluée (6), listes face à la baseline (7), probabilités (8), comparaison avec la référence (9), favoris peu observés (10), conclusion (11).

# 2. Préparation

Le notebook n'utilise que des fonctions du paquet `access_it`. Le journal n'affiche que les avertissements. Jupyter est lancé depuis le dossier `notebooks/`.

In [1]:
import logging
import time
from pathlib import Path

import pandas as pd

from access_it.analytics.race_matching import match_races
from access_it.common.log import setup_logging
from access_it.config import get_settings
from access_it.etl.extract.cache import SNAPSHOTS_DIR
from access_it.export.s3 import download_latest_snapshot
from access_it.export.snapshot import read_snapshot_zip
from access_it.export.upcoming import FORECASTS_KEYS, download_upcoming_entries_and_forecasts
from access_it.model.evaluation import compare_race_metrics, select_evaluated_forecasts
from access_it.model.forecasts import RANK_COLUMNS
from access_it.model.metrics import TOP_N, summarize_metrics
from access_it.model.target_metrics import (
    LIST_METRICS, compute_calibration, compute_list_metrics, compute_probability_metrics
)
from access_it.model.upcoming import forecast_upcoming_entries

setup_logging(logging.WARNING)
pd.set_option("display.width", 120)

Constantes : dossier local des paires téléchargées (ignoré par Git), premier et dernier vendredi des paires rétroactives, période évaluée, et tranches du nombre de courses récentes utilisées (`nb_races_used`, section 10).

In [2]:
UPCOMING_DIR = Path("data/upcoming")
FIRST_FRIDAY, LAST_FRIDAY = "2026-02-13", "2026-10-09"
TEST_START, TEST_END = "2026-01-01", "2026-12-31"
NB_RACES_USED_BINS = [0, 1, 2, 3, 6, 11]
NB_RACES_USED_LABELS = ["0", "1", "2", "3-5", "6-10"]

# 3. Données

Les tables `races` et `rankings` viennent du dernier instantané de la base sur le bucket. Les paires sont téléchargées depuis le bucket, puis seules les 35 paires rétroactives sont gardées : les paires des exécutions programmées ultérieures ne sont pas évaluées ici.

In [3]:
bucket = get_settings().s3_bucket
snapshot_path = download_latest_snapshot(bucket, SNAPSHOTS_DIR)
tables = read_snapshot_zip(snapshot_path)
races = tables["races"]
rankings = tables["rankings"]

In [4]:
fridays = pd.date_range(FIRST_FRIDAY, LAST_FRIDAY, freq="7D", tz="UTC")
runs = sorted(
    (run for run in download_upcoming_entries_and_forecasts(bucket, UPCOMING_DIR)
     if pd.Timestamp(run.timestamp) in fridays),
    key=lambda run: run.timestamp,
)
if len(runs) != len(fridays) or any(run.entries_path is None for run in runs):
    raise ValueError("Missing retroactive pairs on the bucket")
entries = pd.concat([pd.read_parquet(run.entries_path) for run in runs], ignore_index=True)
forecasts = pd.concat([pd.read_parquet(run.forecasts_path) for run in runs], ignore_index=True)
print(f"{len(runs)} pairs, from {FORECASTS_KEYS.build_key(runs[0].timestamp)}"
      f" to {FORECASTS_KEYS.build_key(runs[-1].timestamp)}")

35 pairs, from forecasts/forecasts_20260213T000000Z.parquet to forecasts/forecasts_20261009T000000Z.parquet


Contrôle : les engagés et les prévisions des 35 paires doivent donner 377 courses et 21 207 engagés, sans doublon, toutes les courses ayant une prévision.

In [5]:
print(f"Entries: {entries['race_id'].nunique()} races, {len(entries)} riders entered")
print(f"Forecasts: {forecasts['race_id'].nunique()} races, {len(forecasts)} riders forecast")
print("Duplicated entries:", entries.duplicated(["race_id", "rider_id"]).sum())
print("Forecast races missing from the entries:",
      len(set(forecasts["race_id"]) - set(entries["race_id"])))

Entries: 377 races, 21207 riders entered
Forecasts: 377 races, 21207 riders forecast
Duplicated entries: 0
Forecast races missing from the entries: 0


#### Lecture

Les données sont complètes : 377 courses, 21 207 engagés, aucun doublon, et toutes les courses ont une prévision. On peut maintenant chercher, pour chaque course des engagés, son classement officiel.

# 4. Rapprochement avec les classements officiels

Une course des engagés n'a aucun lien direct avec les courses de la base. `match_races` les rapproche par les coureurs qu'elles partagent : une course officielle est retenue si au moins la moitié de ses classés figurent parmi les engagés, en cherchant du jour de la course à deux jours plus tard. Le tableau donne le nombre de courses des engagés selon le nombre de courses officielles rapprochées.

In [6]:
matches = match_races(entries, races, rankings)
nb_ffc_races = matches.groupby("race_id")["ffc_race_id"].count()
print(nb_ffc_races.value_counts().sort_index().rename_axis("ranked races").rename("races"))
simple_matches = matches.loc[
    matches["race_id"].isin(nb_ffc_races.index[nb_ffc_races == 1]), ["race_id", "ffc_race_id"]
]

ranked races
0     15
1    309
2     53
Name: races, dtype: int64


#### Lecture

- **309 courses simples** sont rapprochées d'un seul classement officiel : ce sont elles qui sont évaluées.
- **53 courses doubles** (un tableau d'engagés pour deux classements) sont écartées.
- **15 courses sans correspondance** sont écartées : annulations, courses regroupées avec l'Open côté FFC, épreuves féminines, course hors des deux régions. Une partie d'entre elles n'est plus extraite depuis les contrôles de courses annulées et de start lists féminines, mais les paires rétroactives sont antérieures.

Ces chiffres sont ceux de la mise au point du rapprochement. Avant d'évaluer, il reste à mesurer l'écart entre les engagés et les classés dans les courses simples.

# 5. Populations

Dans chaque course simple, on distingue trois populations :

- les **engagés classés**, seuls coureurs pour lesquels on dispose à la fois d'une prévision et d'un rang réel ;
- les **engagés non classés** ;
- les **classés non engagés**.

On compte aussi les courses dont le vainqueur, ou au moins un coureur du podium, n'était pas engagé.

In [7]:
official = rankings.rename(columns={"race_id": "ffc_race_id", "finish_rank": "official_rank"})
entered = forecasts.merge(simple_matches, on="race_id").merge(
    official, on=["ffc_race_id", "rider_id"], how="left"
)
ranked = official[official["ffc_race_id"].isin(simple_matches["ffc_race_id"])].merge(
    entered[["ffc_race_id", "rider_id"]], how="left", indicator=True
)
not_entered = ranked[ranked["_merge"] == "left_only"]
nb_entered, nb_ranked = len(entered), len(ranked)
nb_entered_ranked = entered["official_rank"].notna().sum()
print(f"Races: {simple_matches['race_id'].nunique()}")
print(f"Riders entered: {nb_entered}, ranked {nb_entered_ranked}"
      f" ({nb_entered_ranked / nb_entered:.1%}), not ranked {nb_entered - nb_entered_ranked}"
      f" ({1 - nb_entered_ranked / nb_entered:.1%})")
print(f"Riders ranked: {nb_ranked}, not entered {len(not_entered)}"
      f" ({len(not_entered) / nb_ranked:.1%})")
nb_races = simple_matches["ffc_race_id"].nunique()
for name, last_rank in [("winner", 1), ("podium rider", 3)]:
    nb = not_entered.loc[not_entered["official_rank"] <= last_rank, "ffc_race_id"].nunique()
    print(f"Races with a {name} not entered: {nb} ({nb / nb_races:.1%})")

Races: 309
Riders entered: 15663, ranked 12355 (78.9%), not ranked 3308 (21.1%)
Riders ranked: 14328, not entered 1973 (13.8%)
Races with a winner not entered: 46 (14.9%)
Races with a podium rider not entered: 109 (35.3%)


#### Lecture

- **Un engagé sur cinq n'est pas classé** (21,1 %). Cette part mêle les non-partants, les abandons et les coureurs arrivés au-delà des places classées ; les données ne permettent pas de les distinguer.
- **Un classé sur sept n'était pas engagé** (13,8 %). Le vainqueur n'était pas engagé dans 15 % des courses, et au moins un coureur du podium dans 35 % : sans le retrait de ces coureurs, le modèle serait pénalisé dans une course sur sept pour un vainqueur qu'il ne pouvait pas connaître.

On construit maintenant la table évaluée selon les règles de l'introduction.

# 6. Table évaluée

Chaque engagé reçoit son rang officiel ; le rang réel est recalculé parmi les engagés classés (rang « compétition » : les ex aequo gardent le même rang, le suivant saute) ; un engagé non classé reste sans rang. On garde ensuite les courses de plus de 20 engagés classés, puis la période de 2026 avec `select_evaluated_forecasts`.

In [8]:
entered["finish_rank"] = entered.groupby("race_id")["official_rank"].rank(method="min")
nb_ranked_entered = entered.groupby("race_id")["finish_rank"].transform("count")
kept = entered[nb_ranked_entered > TOP_N]
evaluated = select_evaluated_forecasts(kept, TEST_START, TEST_END).reset_index(drop=True)
print(f"Races: {entered['race_id'].nunique()} simple, "
      f"{kept['race_id'].nunique()} with more than {TOP_N} riders entered and ranked, "
      f"{evaluated['race_id'].nunique()} evaluated ({len(evaluated)} riders entered)")
by_race = evaluated.groupby("race_id")["finish_rank"].agg(["size", "count"])
print(f"Per race (median): {by_race['size'].median():.0f} riders entered,"
      f" {by_race['count'].median():.0f} of them ranked")

Races: 309 simple, 282 with more than 20 riders entered and ranked, 282 evaluated (14984 riders entered)
Per race (median): 51 riders entered, 41 of them ranked


#### Lecture

**282 courses** sont évaluées : 27 courses simples sont écartées car elles comptent 20 engagés classés ou moins. Une course évaluée compte en médiane 51 engagés, dont 41 classés.

Pour juger le modèle, il faut aussi un point de comparaison simple : la baseline.

**Baseline approchée.** La baseline du notebook de modélisation (`rank_by_recent_form`) classe les coureurs par taux de top 20 sur leurs 10 dernières courses des 12 derniers mois, puis départage par leurs places successives ; elle ne s'applique qu'aux classés d'une course. Les prévisions stockées contiennent les mêmes features, calculées avec les mêmes réglages, mais pas la liste des places. La baseline est donc approchée : taux de top 20 décroissant, puis meilleure place, place moyenne et `rider_id`. Seul le départage des ex aequo diffère. Son rang sert pour les quatre cibles.

In [9]:
baseline = evaluated.sort_values(
    ["race_id", "top_n_rate", "best_place", "mean_place", "rider_id"],
    ascending=[True, False, True, True, True],
)
baseline = baseline.assign(
    **{rank: baseline.groupby("race_id").cumcount() + 1 for rank in RANK_COLUMNS}
)

# 7. Listes : modèle et baseline

Moyennes des métriques de listes sur les 282 courses, puis écart course par course entre le modèle et la baseline (`compare_race_metrics`) : écart moyen, erreur type et leur rapport (un rapport supérieur à 2 environ, en valeur absolue, indique un écart réel).

In [10]:
model_lists = compute_list_metrics(evaluated)
baseline_lists = compute_list_metrics(baseline)
pd.DataFrame({
    "model": summarize_metrics(model_lists, LIST_METRICS).set_index("metric")["mean"],
    "baseline": summarize_metrics(baseline_lists, LIST_METRICS).set_index("metric")["mean"],
    "nb_races": summarize_metrics(model_lists, LIST_METRICS).set_index("metric")["nb_races"],
}).round(3)

,model,baseline,nb_races
metric,,,
win_recall_at_10,0.567,0.472,282
win_rr_at_10,0.226,0.190,282
podium_recall_at_10,0.551,0.454,282
top5_recall_at_10,0.513,0.440,282
top10_recall_at_20,0.725,0.693,282


In [11]:
compare_race_metrics(model_lists, baseline_lists, LIST_METRICS).round(4)

,metric,difference,std_error,ratio,nb_races
0,win_recall_at_10,0.0957,0.0326,2.9335,282
1,win_rr_at_10,0.0361,0.0179,2.0226,282
2,podium_recall_at_10,0.0969,0.0172,5.6464,282
3,top5_recall_at_10,0.0738,0.0128,5.7707,282
4,top10_recall_at_20,0.0315,0.0063,5.0329,282


#### Lecture

| Liste | Baseline | Modèle | Écart (rapport) |
|---|---|---|---|
| Vainqueur dans la liste de 10 | 0,472 | 0,567 | +0,096 (2,9) |
| Rang réciproque du vainqueur | 0,190 | 0,226 | +0,036 (2,0) |
| Podium dans la liste de 10 | 0,454 | 0,551 | +0,097 (5,6) |
| Top 5 dans la liste de 10 | 0,440 | 0,513 | +0,074 (5,8) |
| Top 10 dans la liste de 20 | 0,693 | 0,725 | +0,032 (5,0) |

- **Le modèle bat nettement la baseline en conditions réelles**, sur toutes les listes : le vainqueur figure dans sa liste de 10 dans 57 % des courses, contre 47 % pour la baseline.
- Les écarts sont du même ordre que dans l'évaluation du notebook de modélisation (section 8.5 : +0,104 pour le vainqueur, +0,097 pour le podium, sur 425 courses avec la baseline exacte) : l'avantage du modèle tient avec les vraies listes d'engagés.

Les listes ne disent rien de la justesse des probabilités affichées : c'est l'objet de la section suivante.

# 8. Probabilités

Log-loss et score de Brier de chaque cible, sur tous les engagés évalués, puis calibration : pour chaque cible, les probabilités sont réparties en 10 groupes de même effectif (déciles), et l'on compare la probabilité moyenne (`predicted`) à la fréquence observée de la cible (`observed`).

In [12]:
compute_probability_metrics(evaluated).round(4)

,target,log_loss,brier,nb_rows
0,win,0.0830,0.0179,14984
1,podium,0.1877,0.0494,14984
2,top5,0.2674,0.0768,14984
3,top10,0.4032,0.1276,14984


In [13]:
compute_calibration(evaluated).pivot(
    index="bin", columns="target", values=["predicted", "observed"]
).round(3)

predicted                      observed                     
target    podium  top10   top5    win   podium  top10   top5    win
bin                                                                
0          0.004  0.024  0.009  0.001    0.006  0.021  0.008  0.001
1          0.008  0.048  0.018  0.002    0.007  0.036  0.017  0.003
2          0.013  0.071  0.028  0.004    0.015  0.059  0.021  0.005
3          0.020  0.096  0.039  0.006    0.026  0.094  0.045  0.007
4          0.028  0.126  0.054  0.009    0.024  0.115  0.051  0.011
5          0.040  0.165  0.073  0.012    0.039  0.152  0.069  0.015
6          0.056  0.212  0.100  0.017    0.048  0.209  0.095  0.011
7          0.079  0.272  0.135  0.024    0.075  0.288  0.128  0.026
8          0.113  0.352  0.184  0.036    0.115  0.374  0.198  0.037
9          0.204  0.517  0.302  0.078    0.209  0.532  0.311  0.073

#### Lecture

**Les probabilités publiées sont bien calibrées.** Dans chaque décile et pour chaque cible, la fréquence observée reste proche de la probabilité moyenne. Pour la victoire, le décile des favoris est prévu à 7,8 % et observé à 7,3 % ; pour le top 10, à 51,7 % et 53,2 %.

Reste à savoir ce que coûte le passage des classés aux vrais engagés : c'est la comparaison avec la référence.

# 9. Comparaison avec la référence

La moyenne de la référence (60,5 %) porte sur les 425 courses de 2026 ; la comparer aux 282 courses évaluées ici mêlerait l'effet des vrais engagés et celui du choix des courses. La référence est donc recalculée sur les mêmes courses : la même fonction `forecast_upcoming_entries`, avec pour engagés les classés officiels de chaque course. Seule la liste des engagés change entre les deux prévisions. Les deux tables sont ensuite comparées course par course.

In [14]:
evaluated_ffc = races.loc[
    races["race_id"].isin(evaluated["ffc_race_id"]), ["race_id", "race_date"]
]
reference_entries = rankings.merge(evaluated_ffc, on="race_id")[
    ["race_id", "race_date", "rider_id", "finish_rank"]
]
start = time.perf_counter()
reference = forecast_upcoming_entries(races, rankings, reference_entries)
print(f"Reference forecast in {time.perf_counter() - start:.0f} s")
reference = reference.rename(columns={"race_id": "ffc_race_id"}).merge(
    simple_matches, on="ffc_race_id"
)
print(f"Races: {reference['race_id'].nunique()} forecast out of {evaluated['race_id'].nunique()},"
      f" {len(reference)} riders ranked")

Reference forecast in 29 s
Races: 282 forecast out of 282, 13723 riders ranked


In [15]:
reference_lists = compute_list_metrics(reference)
pd.DataFrame({
    "real_conditions": summarize_metrics(model_lists, LIST_METRICS).set_index("metric")["mean"],
    "reference": summarize_metrics(reference_lists, LIST_METRICS).set_index("metric")["mean"],
}).round(3)

,real_conditions,reference
metric,,
win_recall_at_10,0.567,0.589
win_rr_at_10,0.226,0.227
podium_recall_at_10,0.551,0.561
top5_recall_at_10,0.513,0.526
top10_recall_at_20,0.725,0.732


In [16]:
compare_race_metrics(model_lists, reference_lists, LIST_METRICS).round(4)

,metric,difference,std_error,ratio,nb_races
0,win_recall_at_10,-0.0213,0.0194,-1.0958,282
1,win_rr_at_10,-0.0012,0.0127,-0.0953,282
2,podium_recall_at_10,-0.0106,0.0118,-0.9042,282
3,top5_recall_at_10,-0.0121,0.0088,-1.3765,282
4,top10_recall_at_20,-0.0071,0.0066,-1.0818,282


In [17]:
pd.concat({
    "real_conditions": compute_probability_metrics(evaluated).set_index("target"),
    "reference": compute_probability_metrics(reference).set_index("target"),
}, axis=1).round(4)

real_conditions                 reference                
              log_loss   brier nb_rows  log_loss   brier nb_rows
target                                                          
win             0.0830  0.0179   14984    0.0900  0.0196   13723
podium          0.1877  0.0494   14984    0.2021  0.0537   13723
top5            0.2674  0.0768   14984    0.2869  0.0830   13723
top10           0.4032  0.1276   14984    0.4301  0.1375   13723

#### Lecture

| Liste | Référence | Conditions réelles | Écart (rapport) |
|---|---|---|---|
| Vainqueur dans la liste de 10 | 0,589 | 0,567 | −0,021 (−1,1) |
| Rang réciproque du vainqueur | 0,227 | 0,226 | −0,001 (−0,1) |
| Podium dans la liste de 10 | 0,561 | 0,551 | −0,011 (−0,9) |
| Top 5 dans la liste de 10 | 0,526 | 0,513 | −0,012 (−1,4) |
| Top 10 dans la liste de 20 | 0,732 | 0,725 | −0,007 (−1,1) |

- **Les vraies listes d'engagés ne dégradent pas les listes de façon mesurable** : les écarts vont de 0 à 2 points, et leurs rapports restent inférieurs à 2 en valeur absolue.
- Sur ces 282 courses, la référence trouve le vainqueur dans 58,9 % des cas, contre 60,5 % sur les 425 courses de 2026 : le choix des courses pèse autant que le passage aux vraies listes d'engagés.
- **Le log-loss et le score de Brier ne sont pas comparables** entre les deux tables : les engagés non classés ajoutent 21 % de lignes sans cible atteinte, faciles à prévoir, ce qui abaisse mécaniquement les deux scores en conditions réelles. La comparaison des probabilités repose sur la calibration (section 8).

# 10. Favoris peu observés

Une limite avait été relevée à la mise au point de la prévision : un coureur avec un seul bon résultat dans sa fenêtre de 12 mois peut passer favori devant des coureurs très réguliers. On regarde ici si elle coûte quelque chose, selon le nombre de courses récentes utilisées pour ses features (`nb_races_used`, de 0 à 10).

D'abord, pour tous les engagés évalués de chaque tranche : probabilités moyennes de victoire et de top 10, et fréquences observées.

In [18]:
evaluated["nb_races_bin"] = pd.cut(
    evaluated["nb_races_used"], bins=NB_RACES_USED_BINS, right=False, labels=NB_RACES_USED_LABELS
)
by_bin = evaluated.groupby("nb_races_bin", observed=True)
pd.DataFrame({
    "riders": by_bin.size(),
    "p_win": by_bin["p_win"].mean(),
    "win": by_bin["finish_rank"].apply(lambda s: (s <= 1).mean()),
    "p_top10": by_bin["p_top10"].mean(),
    "top10": by_bin["finish_rank"].apply(lambda s: (s <= 10).mean()),
}).round(4)

,riders,p_win,win,p_top10,top10
nb_races_bin,,,,,
0,962,0.0183,0.0125,0.1225,0.1019
1,864,0.0173,0.0093,0.1442,0.1169
2,869,0.0171,0.0219,0.1493,0.1507
3-5,2455,0.0188,0.0224,0.1636,0.1633
6-10,9834,0.0192,0.0191,0.2081,0.2125


#### Lecture

- À partir de 2 courses, les probabilités moyennes sont très proches des fréquences observées, pour la victoire comme pour le top 10.
- **Les coureurs peu observés sont un peu surestimés** : avec une seule course, 1,7 % de victoire prévue pour 0,9 % observée (8 victoires pour 15 attendues sur 864 engagés), et 14,4 % de top 10 prévus pour 11,7 % observés ; sans aucune course, 12,3 % de top 10 prévus pour 10,2 % observés. Les effectifs restent faibles, mais le sens de l'écart est celui de la limite redoutée.

La question porte surtout sur les favoris. Pour le rang 1 en victoire, puis pour les rangs 1 à 3 du podium, le tableau donne par tranche le nombre de favoris, le nombre de succès attendu (somme des probabilités), le nombre de succès réels et la part de favoris non classés, en conditions réelles et dans la référence.

In [19]:
reference["nb_races_bin"] = pd.cut(
    reference["nb_races_used"], bins=NB_RACES_USED_BINS, right=False, labels=NB_RACES_USED_LABELS
)


def favourites(rows: pd.DataFrame, probability: str, last_rank: int) -> pd.DataFrame:
    """Favourites by bin of nb_races_used: number, expected and actual tops, share not ranked."""
    groups = rows.groupby("nb_races_bin", observed=True)
    return pd.DataFrame({
        "riders": groups.size(),
        "expected": groups[probability].sum(),
        "actual": groups["finish_rank"].apply(lambda s: (s <= last_rank).sum()),
        "not_ranked": groups["finish_rank"].apply(lambda s: s.isna().mean()),
    })


for name, rank, probability, last_rank, nb_ranks in [
    ("Rank 1 for the win", "rank_win", "p_win", 1, 1),
    ("Ranks 1 to 3 for the podium", "rank_podium", "p_podium", 3, 3),
]:
    print(name)
    print(pd.concat({
        "real_conditions": favourites(evaluated[evaluated[rank] <= nb_ranks], probability, last_rank),
        "reference": favourites(reference[reference[rank] <= nb_ranks], probability, last_rank)
            .drop(columns="not_ranked"),
    }, axis=1).round(3))
    print()

Rank 1 for the win
             real_conditions                            reference                
                      riders expected actual not_ranked    riders expected actual
nb_races_bin                                                                     
0                          1    0.103      0      0.000         1    0.093      0
1                         22    3.498      2      0.136        28    4.452      3
2                         27    3.886      8      0.111        28    4.438      7
3-5                       83   12.531     12      0.193        89   12.788     10
6-10                     149   16.047     10      0.101       136   14.943      9

Ranks 1 to 3 for the podium
             real_conditions                            reference                
                      riders expected actual not_ranked    riders expected actual
nb_races_bin                                                                     
1                         44   12.607     10      

#### Lecture

- **Les favoris avec une seule course ne sont pas un problème mesurable** : rares (22 sur 282 en victoire), ils gagnent 2 fois pour 3,5 attendues, et font 10 podiums pour 12,6 attendus ; les effectifs sont trop petits pour conclure à un écart.
- **Les favoris réguliers (6 à 10 courses) gagnent moins que prévu** : 10 victoires pour 16 attendues. L'écart se retrouve dans la référence (9 pour 14,9) : il ne vient donc pas des conditions réelles. Il reste de l'ordre d'une fluctuation, et ne se retrouve pas pour le podium (131 pour 125 attendus).
- Une partie des échecs des favoris vient de favoris non classés : 10 à 19 % dans les tranches de 1 à 10 courses.

Les favoris peu observés ne coûtent rien de mesurable ; la légère surestimation des coureurs avec une seule course, vue sur l'ensemble des engagés, reste à surveiller. Une correction éventuelle devrait être réglée sur 2023-2025, dans le notebook de modélisation, pour ne pas utiliser les données de 2026 qui servent à la mesure.

# 11. Conclusion

**Mesure.** Sur 282 courses de 2026 prévues à partir des vraies listes d'engagés, classés non engagés retirés du classement officiel et engagés non classés gardés :

- le vainqueur figure dans la liste de 10 du modèle dans 57 % des courses, contre 47 % pour la baseline ; le modèle bat la baseline sur toutes les listes ;
- les probabilités publiées sont bien calibrées ;
- face à la référence recalculée sur les mêmes courses (classés pris comme engagés), aucun écart n'est significatif : les vraies listes d'engagés ne dégradent pas la prévision de façon mesurable.

**Populations.** Un engagé sur cinq n'est pas classé ; un classé sur sept n'était pas engagé, dont le vainqueur dans 15 % des courses.

**Coureurs peu observés.** Les favoris avec une seule course ne posent pas de problème mesurable. Sur l'ensemble des engagés, ceux qui n'ont qu'une course récente sont un peu surestimés (8 victoires pour 15 attendues) : piste à surveiller, à traiter le cas échéant sur la période de réglage 2023-2025.

**Limites.**

- Les 53 courses doubles (un tableau d'engagés pour deux classements) ne sont pas évaluées : le modèle les prévoit comme une seule course.
- Les classés non engagés mêlent inscriptions sur place et engagés non identifiés ; les engagés non classés mêlent non-partants, abandons et arrivés au-delà des places classées.
- La baseline est approchée (départage des ex aequo par la meilleure place et la place moyenne).
- Les paires rétroactives seront effacées du bucket par la rotation à partir de 2027 : le notebook ne pourra plus être relancé tel quel, ses sorties restent.